# Loading Customer.csv into MySQL

This notebook reads `data/orders/Customer.csv` and writes it into the `iem4723_mysql_tutorial` database running in the `mysql` container defined in `docker-compose.yml`.

Run this notebook from inside the `jupyter` service (it already has `pymysql` and `sqlalchemy` installed, and connects to the `mysql` service by its container name). If you are instead running this notebook directly on your host machine, change `DB_HOST` below to `localhost`, since port 3306 is also published to the host.

In [5]:
import pandas as pd
from sqlalchemy import create_engine

**REMEMBER:** You need to connect to the database instance thats running on one of your container images. Therefore, you need to provide details to Python on how to connect to it. This is similar to netowrking and ports. In this case we provide 3306 as our port since thats the port our MySQL instance is *listening* on. 

In [6]:
DB_HOST = "mysql"          # use "localhost" if running outside the jupyter container
DB_PORT = 3306
DB_USER = "root"
DB_PASSWORD = "OSUGoPokes"
DB_NAME = "iem4723_mysql_tutorial"

This is the entire connection string which enables you to connect to the mysql instance. **Engine** represents a variable which holds our connection to the DB instance so that we can use it later. We will come back to it.

In [7]:
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

We have ensured that the `data` folder is already volume mounted. As a result, all the files in data should technically show up here.

In [8]:
csv_path = "../data/orders/Customer.csv"

# Customer_ZIP and Customer_Phone are read as strings so leading zeros
# and formatting are preserved, matching how they should be stored (CHAR/VARCHAR, not INT).
customer_df = pd.read_csv(
    csv_path,
    dtype={"Customer_ZIP": str, "Customer_Phone": str},
)
customer_df.columns = [c.lower() for c in customer_df.columns]

customer_df

,customer_num,customer_firstname,customer_lastname,customer_address,customer_city,customer_state,customer_zip,customer_phone,customer_email
0,C25,Bob,Felton,100 CRYSTAL A DRIVE,HERSHEY,PA,17033,2482604235,Bob@Felton.com
1,C2501,John,Grant,225 WEST STATION SQUARE DRIVE,PITTSBURGH,PA,15219,7817993378,John@grant.com
2,C35,Jill,Young,500 FRANK W BURR BOULEVARD,TEANECK,NJ,7666,9035970279,Jill@Young.com
3,C493,Russ,Jackson,500 WEST MADISON STREET,CHICAGO,IL,60661,8788658690,Russ@Young.com
4,C658,Chris,George,7500 DALLAS PARKWAY,PLANO,TX,75024,9778250256,Chris@George.com
5,C71,Jane,Nicks,2200 PENNSYLVANIA AVENUE NW,WASHINGTON,DC,20037,5694423796,Jane@nicks.com
6,C866,Sam,Stevenson,250 PARKCENTER BOULEVARD,BOISE,ID,83706,3593635559,Sam@Stevenson.com
7,C885,Jane,Newell,3600 LAS VEGAS BOULEVARD SOUTH,LAS VEGAS,NV,89109,2924803403,Jane@Newell.com
8,C938,Rick,Mason,1 VALERO WAY,SAN ANTONIO,TX,78249,6962953646,Rick@Mason.com
9,C959,Lilly,Walters,1025 WEST NASA BOULEVARD,MELBOURNE,FL,32919,9132816703,Lilly@Watson.com


Now you would need to provide columns and define what type of data does our columns hold. Notice how we have used VARCHAR everywhere since we dont really need to carry out any *arithmetic* over any of these fields.

In [ ]:
from sqlalchemy import VARCHAR

column_types = {
    "customer_num": VARCHAR(10),
    "customer_firstname": VARCHAR(50),
    "customer_lastname": VARCHAR(50),
    "customer_address": VARCHAR(100),
    "customer_city": VARCHAR(50),
    "customer_state": VARCHAR(2),
    "customer_zip": VARCHAR(10),
    "customer_phone": VARCHAR(15),
    "customer_email": VARCHAR(100),
}

Now we have a reference to customer file that we just read using Pandas in Python. Once we do this, we can directly push the table on to our MySQL Database instance.

In [ ]:

customer_df.to_sql(
    "customer",
    con=engine,
    if_exists="replace",
    index=False,
    dtype=column_types,
)

Now we are ready to execute our query. Whats our query? To insert all the data rows we have read from file into the MySQL Database. We do this by using the following variables which we have already set:
    1. *engine*: holds the connection string to our already running database instance inside a container
    2. *customer_num*: uniquely identifies each customer.
    3. *conn*: helps execute the query

In [ ]:

# customer_num uniquely identifies each row, so set it as the primary key.
with engine.begin() as conn:
    conn.exec_driver_sql(
        "ALTER TABLE customer ADD PRIMARY KEY (customer_num);"
    )

print(f"Wrote {len(customer_df)} rows to the customer table.")

In [ ]:
# Verify by reading the data back from MySQL
pd.read_sql("SELECT * FROM customer;", con=engine)

## Replicating the phpMyAdmin Steps in Python

Remember everything we just did for customer through phpMyAdmin? Creating a table, inserting a row by hand, importing a CSV, filtering with WHERE and AND? Turns out we can do all of that straight from Python too, using the exact same *engine* we already built above. We will walk through the same steps again below, except this time using Vendor.csv instead of Customer.csv, just so you can see the same tricks work on a completely different file.

In [ ]:
create_vendor_manual = """
DROP TABLE IF EXISTS vendor_manual;
CREATE TABLE vendor_manual (
    Vendor_Num VARCHAR(10) PRIMARY KEY,
    Vendor_Name VARCHAR(50),
    Vendor_City VARCHAR(50),
    Vendor_State VARCHAR(2)
);
"""

with engine.begin() as conn:
    for statement in create_vendor_manual.strip().split(";"):
        if statement.strip():
            conn.exec_driver_sql(statement)

print("vendor_manual table created.")

### Inserting Data via SQL

Remember the INSERT INTO ... VALUES statement we ran directly in phpMyAdmin's SQL tab? Same idea here. We write out the exact same statement as a plain string and run it through *engine*, the same way we just ran CREATE TABLE above.

In [ ]:
insert_vendor = """
INSERT INTO vendor_manual (Vendor_Num, Vendor_Name, Vendor_City, Vendor_State)
VALUES ('V1107', 'CALATLANTIC GROUP', 'ARLINGTON', 'VA');
"""

with engine.begin() as conn:
    conn.exec_driver_sql(insert_vendor)

pd.read_sql("SELECT * FROM vendor_manual;", con=engine)

### Importing a CSV File

How did phpMyAdmin's Import tab build a table for us without us writing any SQL? It just read the CSV's header row and figured out the columns on its own. *pandas* can do the exact same trick. *read_csv* plus *to_sql* builds the vendor table straight from Vendor.csv, and this time we never write CREATE TABLE by hand at all. Just like with customer, Vendor_ZIP and Vendor_Phone are read in as strings so we dont lose any leading zeros or formatting.

We are using the following pieces to make this happen:
    1. *vendor_df*: the DataFrame holding everything we just read from Vendor.csv
    2. *engine*: our already open connection to the database instance inside the container
    3. *Vendor_Num*: uniquely identifies each vendor, so we set it as the primary key once the table exists

In [ ]:
vendor_df = pd.read_csv(
    "../data/orders/Vendor.csv",
    dtype={"Vendor_ZIP": str, "Vendor_Phone": str},
)

vendor_df.to_sql("vendor", con=engine, if_exists="replace", index=False)

# pandas doesn't know Vendor_Num should be a key, so it creates it as TEXT.
# MySQL needs a key length on TEXT columns used in a primary key.
with engine.begin() as conn:
    conn.exec_driver_sql("ALTER TABLE vendor ADD PRIMARY KEY (Vendor_Num(10));")

print(f"Imported {len(vendor_df)} rows into the vendor table.")

### Querying a Table

Remember running SELECT * FROM ... WHERE 1 in phpMyAdmin's SQL tab? That just returned every row in the table. Whats the Python equivalent? *pd.read_sql* runs the exact same query, except now the result comes back as a DataFrame instead of an HTML grid.

In [ ]:
pd.read_sql("SELECT * FROM vendor;", con=engine)

### Filtering with WHERE

Remember adding Item_Price >= 3 in the SQL tab earlier to filter our results? Same idea here. We filter vendor down to only the vendors based in Texas.

In [ ]:
pd.read_sql("SELECT * FROM vendor WHERE Vendor_State = 'TX';", con=engine)

### Combining Conditions with AND

Remember filtering for items priced at three dollars or more that were also Slacks? Same trick again. We narrow our Texas vendors down even further to only the ones based in Dallas.

In [ ]:
pd.read_sql(
    "SELECT * FROM vendor WHERE Vendor_State = 'TX' AND Vendor_City = 'DALLAS';",
    con=engine,
)